### Pinecone


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
#!uv add pinecone

In [3]:
import os
from pinecone import Pinecone, ServerlessSpec

In [ ]:
#  ***** Pinecone_인덱스생성.png 확인하기 *****

# API Key 설정
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "quickstart-index"

# 인덱스가 없는 경우 새로 생성 (OpenAI text-embedding-3-small 기준 1536차원)
if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# 인덱스 연결 
index = pc.Index(index_name) 

In [5]:
# --------------- 한개 테스트 ---------------
# # 예시 벡터 데이터 (id, vector, metadata)
# # * 실제 서비스에서는 임베딩 모델(OpenAI 등)을 통해 생성된 벡터를 넣어야 합니다.
# dummy_vector = [0.01] * 1536  # 1536차원 가상 벡터

# vectors = [
#     {
#         "id": "doc1",
#         "values": dummy_vector,
#         "metadata": {"text": "Pinecone은 벡터 데이터베이스입니다.", "category": "tech"}
#     }
# ]

# # 데이터 업로드 (Upsert)
# index.upsert(vectors=vectors, namespace="example-namespace")

In [ ]:
#  *****  Pinecone_인덱스upsert.png 이미지 확인하기 *****

from openai import OpenAI
import time

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 1. 테스트할 문서
documents = [
    {"id": "doc1", "text": "Pinecone은 벡터 데이터베이스입니다."},
    {"id": "doc2", "text": "Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다."},
]

# 2. 문장을 벡터로 변환해서 Pinecone에 저장
response = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=[document["text"] for document in documents],
)

vectors = [
    {
        "id": document["id"],
        "values": embedding.embedding,
        "metadata": {"text": document["text"]},
    }
    for document, embedding in zip(documents, response.data)
]
# 동작 방식 (Upsert = Update + Insert) : id중복 요청시 오류없이 덮어씀.
index.upsert(vectors=vectors)
time.sleep(2)

# 3. 검색용 질문도 같은 방식으로 벡터 변환
question = "임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?"
question_vector = openai_client.embeddings.create(
    model="text-embedding-3-small",
    input=question,
).data[0].embedding

# 4. 질문과 의미가 가까운 문서 검색
result = index.query(
    vector=question_vector,
    top_k=2,
    include_metadata=True,
)

print(f"질문: {question}\n") 
for match in result.matches:
    print(f"{match.id}: {match.metadata['text']} (유사도: {match.score:.4f})") 

질문: 임베딩 데이터를 저장하고 유사한 정보를 찾는 서비스는 무엇인가요?

doc2: Pinecone은 임베딩 벡터를 저장하고 유사한 정보를 검색하는 서비스입니다. (유사도: 0.6642)
doc1: Pinecone은 벡터 데이터베이스입니다. (유사도: 0.4800)
